# Create CDTI Awards

Creates awards for **CDTI** -- Centro para el Desarrollo Tecnológico y la Innovación E.P.E.
(until 2023 "...Tecnológico Industrial"), Spain's public business R&D+i agency -- from CDTI's
own open-data file **"Proyectos Aprobados"** (dataset *Ayudas Parcialmente Reembolsables y
Subvenciones CDTI* on datos.gob.es), downloaded from
https://www.cdti.es/datos-abiertos-creditos-subvenciones-y-lineas via its JSON API
(`sede.cdti.gob.es/.../DatosAbiertos/api/datos/proyectosidi/JSON/{year}/ALL`).

**Coverage:** every aid approved 2014-2026 (the API offers no year before 2014): **20,614 awards**,
one row per beneficiary file (cooperative projects -- CIEN, Innterconecta, I+D en Cooperación --
appear once per partner, each with its own CDTI aid). Beneficiaries are companies (98.6%),
technology centres, research associations, a few universities and public bodies.

**Loan vs grant (`funding_type`, and the instrument in `funder_scheme`):**
- `TipoAyuda = 'Ayudas Parcialmente Reembolsables'` (partially reimbursable R&D/innovation loans):
  14,876 awards, EUR 8.21 bn -> `funding_type = 'loan'`
- `TipoAyuda = 'Subvención'` (grants): 5,738 awards, EUR 2.37 bn -> `funding_type = 'grant'`
- `funder_scheme = '<InstrumentoFinanciero> - <TipoAyuda>'`, e.g.
  `Proyectos CDTI de I+D - Ayudas Parcialmente Reembolsables`, `Neotec - Subvención`
  (just `<TipoAyuda>` for the 274 rows with no instrument).

**Scope filter: none.** Company R&D funding is in scope (2026-10-01 batch rule). Kept and flagged:
`Líneas de Innovación (LIC, LICa y LIEE)` (2,727 innovation loans: technology adoption /
process innovation rather than R&D) and `Proyectos de Inversión Innovadora FEMPA` (72 loans for
innovative fish-processing investment).

**amount / currency:** `AportacionCDTI` (CDTI's contribution, EUR; for loans the loan amount),
99.7% coverage: the 69 files whose budget and contribution both read `0,00 €` (mostly 2015-2019
Innterconecta / Eurostars) get NULL. The project budget (`Presupuesto`) stays in the raw table.

**lead_investigator:** the file names no people; the lead carries only the beneficiary
organisation (`RazonSocial`, country ES) as affiliation, names NULL (HHMI/Mott precedent).
No description is published; `description` is NULL.

**`funder_award_id` (§2.1.1):** papers cite CDTI file numbers (`IDI-20180224`, `MIG-20221048`,
`CER-20191002`; 1,588 priority-0 citation rows for F4320321043), but the open-data file does not
carry them. We ship a stable synthetic key `CDTI-{NIF}-{YYYYMMDD approval}-{md5(title)[:10]}`
(plus `-{postcode}` for the 2 company/date/title pairs filed twice). These keys do not collapse
onto citation stubs.

**Prerequisites:** run `scripts/local/cdti_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/cdti/cdti_projects.parquet` (§1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320321043`, Centro para el Desarrollo
Tecnológico Industrial, ES; ror 0124x7055 and doi 10.13039/501100001872 read from
`openalex.funders.funders`.

**Priority:** `572` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cdti_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/cdti/cdti_projects.parquet`;

In [ ]:
%sql
SELECT TipoAyuda, COUNT(*) as n, ROUND(SUM(TRY_CAST(amount_cdti_eur AS DOUBLE)), 0) as eur
FROM openalex.awards.cdti_raw
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.cdti_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.cdti_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320321043 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'CDTI F4320321043 must have exactly one openalex.funders.funders row') as funder_ok,
       first(display_name) as display_name, first(ror_id) as ror_id, first(doi) as doi, first(country_code) as country_code
FROM openalex.funders.funders
WHERE funder_id = 4320321043;

## Step 2: Create CDTI Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cdti_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320321043  -- CDTI
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.TituloProyecto), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    -- 69 files (mostly 2015-2019 Innterconecta / Eurostars) show budget and contribution 0,00: NULL, not 0
    NULLIF(TRY_CAST(g.amount_cdti_eur AS DOUBLE), 0) as amount,
    CASE WHEN NULLIF(TRY_CAST(g.amount_cdti_eur AS DOUBLE), 0) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.TipoAyuda = 'Subvención' THEN 'grant'
         WHEN g.TipoAyuda = 'Ayudas Parcialmente Reembolsables' THEN 'loan'
         ELSE 'grant' END as funding_type,
    CASE WHEN NULLIF(TRIM(g.InstrumentoFinanciero), '') IS NOT NULL
         THEN CONCAT(TRIM(g.InstrumentoFinanciero), ' - ', TRIM(g.TipoAyuda))
         ELSE NULLIF(TRIM(g.TipoAyuda), '') END as funder_scheme,
    'cdti_open_data' as provenance,
    TRY_TO_DATE(g.approval_date, 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    YEAR(TRY_TO_DATE(g.approval_date, 'yyyy-MM-dd')) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN NULLIF(TRIM(g.RazonSocial), '') IS NOT NULL THEN named_struct(
        'given_name', CAST(NULL AS STRING),
        'family_name', CAST(NULL AS STRING),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', TRIM(g.RazonSocial),
            'country', 'ES',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    'https://www.cdti.es/datos-abiertos-creditos-subvenciones-y-lineas' as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.cdti_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- §2.1.1 shape check (fails the run BEFORE the insert): every key is the synthetic
-- CDTI-{NIF}-{YYYYMMDD}-{md5[:10]}[-{postcode}] form; ids unique; all rows carried over.
SELECT
    assert_true(COUNT_IF(NOT funder_award_id RLIKE '^CDTI-[A-Z][0-9]{7}[0-9A-J]-[0-9]{8}-[0-9a-f]{10}(-[0-9]{5})?$') = 0,
                'funder_award_id not in the CDTI-{NIF}-{date}-{hash} form') as shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') as ids_unique,
    assert_true(COUNT(*) = (SELECT COUNT(*) FROM openalex.awards.cdti_raw), 'rows dropped in transform') as all_rows,
    COUNT(*) as total
FROM openalex.awards.cdti_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cdti_open_data' AND priority = 572;

-- Insert into openalex_awards_raw with priority.
-- Priority 572: direct-from-funder ingest of CDTI's open-data file.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    572 as priority
FROM openalex.awards.cdti_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.cdti_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_date,
       lead_investigator.affiliation.name as beneficiary
FROM openalex.awards.cdti_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.cdti_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.cdti_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- §6.4a frequency check: no person names; beneficiary organisations top-20 (repeat R&D borrowers are expected).
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family,
       lead_investigator.affiliation.name AS org, COUNT(*) AS n
FROM openalex.awards.cdti_awards
GROUP BY 1, 2, 3 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.4a title frequency check (cooperative projects repeat their title once per partner).
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.cdti_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.3 / §6.7 coverage (expect 100% title/start_date/beneficiary, 99.7% amount; EUR only).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_beneficiary
FROM openalex.awards.cdti_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs (expected 0: CDTI's cited file numbers are not in the open-data file).
SELECT c.provenance, COUNT(*) as stub_rows
FROM openalex.awards.openalex_awards_raw c
JOIN openalex.awards.cdti_awards t ON t.id = c.id
WHERE c.funder_id = 4320321043 AND c.priority = 0
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cdti_open_data'
GROUP BY provenance, priority;